# Zadanie 2.5: uproszczenia członów równania

Sprawdzamy, które człony równania bazowego są niezbędne, a które można uprościć małym kosztem (issue #13, wejście do #30):

1. **Bez dyfuzji ($D = 0$):** każdy punkt siatki staje się osobnym ODE, znika naciekanie i napływ komórek z brzegu.
2. **Wzrost wykładniczy ($K = \infty$):** usuwamy nasycenie logistyczne $-(\rho/K)u^2$ przy tym samym warunku początkowym $u_0$.
3. **Skok LQ zamiast impulsów (`dose="jump"`):** natychmiastowe mnożenie przez $e^{-\beta R(x)}$ w chwilach frakcji.

Testujemy je na czterech scenariuszach z `zad2_02_scenariusze` (10 frakcji co 0,4 od $t = 1$, solver B, $\Delta x = 0{,}1$).

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.solvers import simulate

rng = seed_everything()

x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)
u0 = initial_condition(x)
after = t >= schedule.times[-1] + 3 * schedule.width

plans = {
    "pole pokrywa guz, duża dawka": Params(beta=1.5, s=0.0, r=20.0),
    "pole pokrywa guz, mała dawka": Params(beta=0.45, s=0.0, r=20.0),
    "pole za wąskie": Params(beta=1.5, s=0.0, r=3.0),
    "pole przesunięte": Params(beta=1.5, s=6.0, r=8.0),
}

variants = {
    "model bazowy": lambda p: simulate(p, schedule, x, t, u0=u0, method="cn", dose="pulse"),
    "bez dyfuzji (D = 0)": lambda p: simulate(replace(p, D=0.0), schedule, x, t, u0=u0, method="cn", dose="pulse"),
    "wzrost wykładniczy (K = ∞)": lambda p: simulate(replace(p, K=np.inf), schedule, x, t, u0=u0, method="cn", dose="pulse"),
    "skok LQ zamiast impulsów": lambda p: simulate(p, schedule, x, t, u0=u0, method="cn", dose="jump"),
}

masses = {(plan, var): tumor_mass(fn(p), x) for plan, p in plans.items() for var, fn in variants.items()}

## Tabela: uproszczenie → błąd

Dla każdego wariantu i scenariusza liczymy błąd względny $M(t)$ na całym przedziale i po zakończeniu frakcji,
błąd względny $M_{\min}$ oraz różnicę czasu do odrostu względem modelu bazowego.

In [ ]:
rows = []
for var in list(variants)[1:]:
    for plan in plans:
        M0_curve = masses[(plan, "model bazowy")]
        M = masses[(plan, var)]
        m_min0, _ = min_mass(t, M0_curve)
        m_min, _ = min_mass(t, M)
        t_reg0 = time_to_regrowth(t, M0_curve)
        t_reg = time_to_regrowth(t, M)
        dt_reg = t_reg - t_reg0 if (np.isfinite(t_reg) and np.isfinite(t_reg0)) else np.nan
        rows.append({
            "uproszczenie": var,
            "scenariusz": plan,
            "max błąd wzgl. M(t)": float(np.max(np.abs(M - M0_curve) / M0_curve)),
            "max błąd wzgl. po leczeniu": float(np.max(np.abs(M[after] - M0_curve[after]) / M0_curve[after])),
            "błąd wzgl. M_min": float(abs(m_min - m_min0) / m_min0),
            "czas odrostu (baza)": t_reg0,
            "czas odrostu (uprosz.)": t_reg,
            "Δ czasu odrostu": dt_reg,
            "M(10) / M_baza(10)": float(M[-1] / M0_curve[-1]),
        })

table = pd.DataFrame(rows)
table.to_csv(TABLES / "zad2_05_uproszczenia.csv", index=False)
table.style.format(precision=3)

In [ ]:
summary = (
    table.groupby("uproszczenie", sort=False)
    .agg(
        mediana_max_bledu_M=("max błąd wzgl. M(t)", "median"),
        max_blad_po_leczeniu=("max błąd wzgl. po leczeniu", "max"),
        max_blad_M_min=("błąd wzgl. M_min", "max"),
        max_abs_dt_odrostu=("Δ czasu odrostu", lambda s: float(np.nanmax(np.abs(s))) if s.notna().any() else np.nan),
        max_stosunek_M10=("M(10) / M_baza(10)", "max"),
    )
    .reset_index()
)
summary.to_csv(TABLES / "zad2_05_uproszczenia_podsumowanie.csv", index=False)
summary.style.format(precision=3)

## Przebiegi masy guza $M(t) / M(0)$

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2), sharex=True)
styles = {
    "model bazowy": ("k-", 2.2),
    "bez dyfuzji (D = 0)": ("C0--", 1.8),
    "wzrost wykładniczy (K = ∞)": ("C3:", 1.8),
    "skok LQ zamiast impulsów": ("C2-.", 1.5),
}
for ax, plan in zip(axes.flat, plans):
    M0 = masses[(plan, "model bazowy")][0]
    for var, (fmt, lw) in styles.items():
        M = masses[(plan, var)]
        ax.semilogy(t, np.maximum(M / M0, 1e-6), fmt, lw=lw, label=var)
    ax.axvspan(schedule.times[0] - 0.1, schedule.times[-1] + 0.1, color="0.93", lw=0)
    ax.set(title=plan, ylabel="M(t) / M(0)")
    ax.grid(alpha=0.25)
for ax in axes[1]:
    ax.set_xlabel("czas t")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_05_uproszczenia.png", dpi=150)

## Wnioski

- **Wzrost logistyczny jest niezbędny.** Zastąpienie go wzrostem wykładniczym ($K = \infty$) zawyża masę końcową $M(10)$
  od 1,7 raza (pełne pokrycie, duża dawka) do 91–1738 razy w scenariuszach z odrostem, bo gęstość nie nasyca się w $K$
  i rośnie bez ograniczenia już przed pierwszą frakcją.
- **Dyfuzja jest niezbędna, szczególnie przy niedopasowanym polu.** Przy $D = 0$ każdy węzeł ewoluuje niezależnie,
  więc guz nie poszerza się między frakcjami i komórki z nienapromienionego brzegu nie wracają w głąb pola.
  Powoduje to systematyczne zaniżenie masy końcowej o 31–50% i opóźnienie czasu do odrostu o 0,40 (mała dawka),
  0,75 (pole za wąskie) i aż **2,35** (7,80 zamiast 5,45) dla pola przesuniętego.
- **Skok LQ zamiast impulsów jest jedynym bezpiecznym uproszczeniem.** Duża różnica chwilowa (25–109%) pojawia się wyłącznie
  dokładnie w chwilach frakcji $t_i$, gdzie skok działa natychmiastowo, a impuls ma szerokość $w = 0{,}05$.
  Po zakończeniu leczenia błąd względny $M(t)$ nie przekracza **0,39%**, błąd $M_{\min}$ wynosi poniżej 4,7%,
  a czas do odrostu różni się najwyżej o 0,15 (1–3 kroki zapisu).
- **Wniosek do Zadania 4 (#30):** redukując model, nie wolno usuwać ani dyfuzji, ani nasycenia logistycznego —
  można natomiast bezpiecznie zastąpić gładkie impulsy skokiem LQ (przyspiesza obliczenia i upraszcza surogaty ODE).